In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from Log_Extractor import LogExtractor
from Fast_Log_Ext import Fast_LogExtractor
import os

In [2]:
# ==========================================
# ⚙️ 1. 설정 및 매개변수 (여기만 바꾸면 14개 펌프 무한 확장 가능!)
# ==========================================
PUMP_ID = 'P1' # 나중에 P2, P3 등으로 변경
TANK_ID = 'P1' # 나중에 T2, T3 등으로 변경
ROBOT_ID = 'RB1' # 나중에 R2, R3 등으로 변경
MODEL_SAVE_DIR = './Ana_models' # 모델과 스케일러를 저장할 폴더
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [3]:
def prepare_pump_features(raw_df, pump_id, tank_id, robot_id):
    """
    raw_df를 받아 해당 펌프(pump_id)의 피처를 생성하여 model_df를 반환합니다.
    """
    df = raw_df.ffill().fillna(0).copy()
    df_reset = df.reset_index() 
    
    # 동적 태그 이름 생성 (P1, P2 등 자동 대응)
    tag_SV = f'g_s_SV_{pump_id}'
    tag_Ana = f'Ana_Out_{pump_id}'
    tag_PT = f'Scale_Out___PT_{pump_id}'
    tag_FT = f'Scale_Out___FT_{pump_id}'
    tag_Temp = f'TK_Temp_PV_{tank_id}'

    # [신규] 완벽한 타이밍을 위한 하드 트리거 태그
    tag_BuildUp = f'Pump_BuildUp_{pump_id}' 
    tag_Wagon = f'{robot_id}_Robot_Num'

    # ==========================================
    # ✂️ 1. 완벽한 칼단발 블록(Block) 나누기
    # ==========================================
    # 대차 번호가 바뀌거나, 펌프 가동 상태(ON/OFF)가 바뀔 때마다 새로운 블록 ID를 부여합니다.
    df_reset['Wagon_Changed'] = df_reset[tag_Wagon].diff() != 0
    df_reset['BuildUp_Changed'] = df_reset[tag_BuildUp].diff() != 0
    df_reset['Wagon_Block_ID'] = (df_reset['Wagon_Changed'] | df_reset['BuildUp_Changed']).cumsum()

    # ==========================================
    # 🎯 2. 진짜 샷 추출 (조건부 필터링 싹 다 폐기)
    # ==========================================
    # BuildUp이 1(가동 중)인 데이터만 남깁니다. 이것이 진짜 샷입니다.
    df_shots = df_reset[df_reset[tag_BuildUp] == 1].copy()

    # ==========================================
    # 🔄 3. 이전 SV 값(Prev_SV) 구하기
    # ==========================================
    # 각 유효 블록별로 최대 SV를 구한 뒤, 이전 샷의 SV를 1칸 당겨옵니다.
    block_summary = df_shots.groupby('Wagon_Block_ID').agg(
        Max_SV=(tag_SV, 'max')
    )
    block_summary['Prev_SV'] = block_summary['Max_SV'].shift(1)

    # 원본 df_shots에 Prev_SV 정보 맵핑
    df_shots = df_shots.merge(block_summary[['Prev_SV']], left_on='Wagon_Block_ID', right_index=True, how='left')
    df_shots = df_shots.sort_index()

    # ==========================================
    # 📈 4. 실시간 피처 엔지니어링 (기존 로직 완벽 유지)
    # ==========================================
    model_df = df_shots.copy()
    model_df['Tick_Index'] = model_df.groupby('Wagon_Block_ID').cumcount()
    model_df['Phase_Start'] = np.where(model_df['Tick_Index'] < 2, 1.0, 0.0)
    model_df['Phase_Steady'] = np.where(model_df['Tick_Index'] >= 2, 1.0, 0.0)

    model_df['Instant_FT_Error'] = model_df[tag_SV] - model_df[tag_FT]
    model_df['Instant_FT_Error_Rate'] = np.where(
        model_df[tag_SV] > 0, 
        (model_df['Instant_FT_Error'] / model_df[tag_SV]) * 100, 
        0
    )
    model_df['Cum_FT_Error'] = model_df.groupby('Wagon_Block_ID')['Instant_FT_Error'].cumsum()

    model_df['Prev_SV_Diff'] = model_df[tag_SV] - model_df['Prev_SV'].fillna(0)
    model_df['Rolling_PT_Max_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].rolling(window=3, min_periods=1).max().reset_index(0, drop=True)
    model_df['Rolling_PT_Diff_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].diff(periods=2).fillna(0)

    model_df['Instant_SV_Diff'] = model_df[tag_SV].diff().fillna(0)
    model_df['Phase_Transition'] = np.where(model_df['Instant_SV_Diff'] != 0, 1.0, 0.0)

    # ==========================================
    # 🧹 5. 최종 컬럼 정리
    # ==========================================
    feature_cols = [
        tag_SV, 'Prev_SV', 'Prev_SV_Diff', 
        tag_Ana, tag_Temp,               
        tag_PT, 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
        tag_FT, 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
        'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
    ]
    
    # 결측치 최종 정리 (Shift나 Rolling에서 발생한 NaN 처리)
    model_df = model_df.fillna(0)
    
    return model_df[feature_cols], feature_cols

print("✅ 동적 피처 생성 함수 준비 완료! (하드 트리거 기반)")

✅ 동적 피처 생성 함수 준비 완료! (하드 트리거 기반)


In [4]:
# 한글 폰트 설정
plt.rc('font', family='AppleGothic') 
plt.rcParams['axes.unicode_minus'] = False

# 1. 추출기 가동 (어제 하루치 데이터만 가져오기)
# extractor는 이미 위에서 선언했다고 가정합니다.
target_tags = [
    f'Pump_BuildUp_{PUMP_ID}' , f'{ROBOT_ID}_Robot_Num', 
    f'g_s_SV_{PUMP_ID}', f'Ana_Out_{PUMP_ID}', 
    f'Scale_Out___PT_{PUMP_ID}', f'Scale_Out___FT_{PUMP_ID}', 
    f'TK_Temp_PV_{TANK_ID}', f'TK_Level_PV_{TANK_ID}'
]
extractor = LogExtractor()
fast_ex = Fast_LogExtractor()

🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!
🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!


In [5]:
from datetime import datetime, timedelta

# ==========================================
# 📅 1. 날짜 범위 설정 (최근 7일)
# ==========================================
end_date = datetime.now() - timedelta(days=2)  # 어제까지가 학습 데이터
start_date = end_date - timedelta(days=3)      # 3일 전부터


# 데이터를 담을 리스트
train_chunks = []

print(f"🔍 {PUMP_ID} 학습 데이터 추출 시작 ({start_date.date()} ~ {end_date.date()})")

# ==========================================
# 🔄 2. 하루 단위 루프 돌리기
# ==========================================
current_date = start_date
while current_date <= end_date:
    # InfluxDB용 시간 포맷팅 (ISO 8601)
    day_start = current_date.strftime('%Y-%m-%dT00:00:00Z')
    day_end = (current_date + timedelta(days=1)).strftime('%Y-%m-%dT00:00:00Z')
    
    print(f"   📅 {current_date.date()} 데이터 요청 중...")
    
    try:
        # 하루치씩 쏙쏙 뽑아오기
        day_raw = fast_ex.get_data(start_time=day_start, end_time=day_end, target_tags=target_tags)
        
        if not day_raw.empty:
            train_chunks.append(day_raw)
            print(f"      ✅ {len(day_raw)}행 수집 완료")
        else:
            print(f"      ⚠️ 데이터가 없습니다.")
            
    except Exception as e:
        print(f"      ❌ 오류 발생: {e}")
    
    current_date += timedelta(days=1)


🔍 P1 학습 데이터 추출 시작 (2026-04-10 ~ 2026-04-13)
   📅 2026-04-10 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-10T00:00:00Z ~ 2026-04-11T00:00:00Z)
✅ 추출 완료! 총 39071행, 8개 컬럼 확보.
      ✅ 39071행 수집 완료
   📅 2026-04-11 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-11T00:00:00Z ~ 2026-04-12T00:00:00Z)
✅ 추출 완료! 총 38099행, 4개 컬럼 확보.
      ✅ 38099행 수집 완료
   📅 2026-04-12 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-12T00:00:00Z ~ 2026-04-13T00:00:00Z)
✅ 추출 완료! 총 42247행, 8개 컬럼 확보.
      ✅ 42247행 수집 완료
   📅 2026-04-13 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-13T00:00:00Z ~ 2026-04-14T00:00:00Z)
✅ 추출 완료! 총 41576행, 8개 컬럼 확보.
      ✅ 41576행 수집 완료


In [ ]:
target_test_tags = [
    Time,
    f'Pump_BuildUp_{PUMP_ID}' , f'{ROBOT_ID}_Robot_Num', 
    f'g_s_SV_{PUMP_ID}', f'Ana_Out_{PUMP_ID}', 
    f'Scale_Out___PT_{PUMP_ID}', f'Scale_Out___FT_{PUMP_ID}', 
    f'TK_Temp_PV_{TANK_ID}', f'TK_Level_PV_{TANK_ID}'
]

# ==========================================
# 🧩 3. 데이터 병합 및 피처 생성
# ==========================================
if train_chunks:
    raw_train = pd.concat(train_chunks)
    print(f"\n✅ 전체 데이터 병합 완료! 총 {len(raw_train)}행")
    
    # 기존 피처 생성 함수 태우기
    train_df, feature_cols = prepare_pump_features(raw_train, PUMP_ID, TANK_ID, ROBOT_ID)
else:
    print("❌ 수집된 데이터가 없어 학습을 진행할 수 없습니다.")

# Validation(오늘치)은 상대적으로 작으므로 기존처럼 한 번에 가져옵니다.
raw_valid = fast_ex.get_data(start_time="-1d", end_time="now()", target_tags=target_test_tags)


✅ 전체 데이터 병합 완료! 총 160993행
🔍 데이터 추출 시작... (-1d ~ now())
✅ 추출 완료! 총 41147행, 8개 컬럼 확보.


In [7]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from Ana_Preprocess import VirtualSensorPreprocessor

print("🏋️‍♂️ 가상 센서 (컨닝 방지 버전) 재학습 시작...")

# 1. 아까 정의한 '순수 물리 피처' 7개
pure_physical_features = [
    'Scale_Out___PT_P1', 
    'Rolling_PT_Max_3', 
    'Rolling_PT_Diff_3', 
    'Scale_Out___FT_P1', 
    'TK_Temp_PV_P1',
    'Phase_Start', 
    'Phase_Steady'
]

# (전처리기 구동을 위해 전체 컬럼 리스트도 필요함)
all_features = [
    'g_s_SV_P1', 'Prev_SV', 'Prev_SV_Diff', 'Ana_Out_P1', 'TK_Temp_PV_P1',               
    'Scale_Out___PT_P1', 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
    'Scale_Out___FT_P1', 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
    'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
]

vs_preprocessor = VirtualSensorPreprocessor(feature_cols=all_features)

# 2. 학습 데이터를 전처리기로 밀어넣어 훈련셋(X, y) 추출
train_records = raw_valid.to_dict('records') # 학습용 데이터(하루치나 일주일치)를 넣으세요!
X_list = []
y_list = []

for row in train_records:
    df_processed, _ = vs_preprocessor.process_raw_tick(row)
    if df_processed is not None:
        # X: 순수 물리 피처 7개만 쏙 뽑기
        X_list.append(df_processed[pure_physical_features].iloc[0].to_dict())
        # y: 정답지 (Ana_Out)
        y_list.append(df_processed.iloc[0]['Ana_Out_P1'])

import pandas as pd
X_train = pd.DataFrame(X_list)
y_train = pd.Series(y_list)

print(f"📊 학습 데이터 준비 완료: {X_train.shape} (7개 피처만 사용!)")

# 3. 모델 학습 (Random Forest)
virtual_sensor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
virtual_sensor.fit(X_train, y_train)

# 4. 저장 (기존 구형 모델 덮어쓰기)
joblib.dump(virtual_sensor, 'Ana_models/virtual_sensor_P1.pkl')
print("✅ 순수 물리 가상 센서 저장 완료! 이제 아까 그 테스트 코드를 다시 돌려보세요!")

🏋️‍♂️ 가상 센서 (컨닝 방지 버전) 재학습 시작...
📊 학습 데이터 준비 완료: (11225, 7) (7개 피처만 사용!)
✅ 순수 물리 가상 센서 저장 완료! 이제 아까 그 테스트 코드를 다시 돌려보세요!


In [9]:
import joblib
from Ana_Preprocess import VirtualSensorPreprocessor

print("🚀 가상 센서 (Virtual Sensor) 단독 테스트 시작...\n")

# 1. 학습된 가상 센서(Random Forest) 로드
# (앞서 Ana_Out을 정답지(y)로 두고 학습한 모델이 있다고 가정합니다)
try:
    virtual_sensor = joblib.load('Ana_models/virtual_sensor_P1.pkl')
    print("✅ 가상 센서 모델 로드 완료!")
except FileNotFoundError:
    print("⚠️ 모델 파일이 없습니다. 코드가 정상 작동하는지 흐름만 테스트합니다.")
    virtual_sensor = None

# 2. 피처 컬럼 세팅 (Ana_Out_P1 제외!)
all_features = [
    'g_s_SV_P1', 'Prev_SV', 'Prev_SV_Diff', 'Ana_Out_P1', 'TK_Temp_PV_P1',               
    'Scale_Out___PT_P1', 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
    'Scale_Out___FT_P1', 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
    'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
]

# 🕵️‍♂️ [핵심] 가상 센서용 순수 물리 피처 (컨닝 방지!)
pure_physical_features = [
    'Scale_Out___PT_P1', 
    'Rolling_PT_Max_3', 
    'Rolling_PT_Diff_3', 
    'Scale_Out___FT_P1', 
    'TK_Temp_PV_P1',
    'Phase_Start', 
    'Phase_Steady'
]

# 전처리기 가동
vs_preprocessor = VirtualSensorPreprocessor(feature_cols=all_features)

# 3. 실시간 스트리밍 시뮬레이션 루프
records = raw_valid.to_dict('records')
loss_alerts = []

for i, live_row in enumerate(records):
    # 전처리기에 1틱 입력
    df_processed, meta_info = vs_preprocessor.process_raw_tick(live_row)
    
    # 가동 중이 아니면(BuildUp == 0) 패스
    if df_processed is None:
        continue
        
    timestamp = live_row.get('Time', 'Unknown Time')

    # PLC가 명령한 원래 Ana_Out 값 추출
    commanded_ana = df_processed.iloc[0]['Ana_Out_P1']
    
    # 🚨 모델 입력 데이터는 '순수 물리 피처'만 칼같이 잘라냅니다!
    X_input = df_processed[pure_physical_features]
    
    if virtual_sensor:
        # 가상 센서 역산: "현재 물리적 결과(PT, FT 등)를 내려면 실제 전류가 얼마여야 하는가?"
        virtual_ana = virtual_sensor.predict(X_input)[0]
        print(f"[{timestamp}] [틱 {meta_info['Tick_Index']:02d}] 명령 Ana: {commanded_ana:.0f} | 가상 역산: {virtual_ana:.0f}")
        
        # 효율 손실률 계산 (%)
        if commanded_ana > 0:
            power_loss_rate = ((commanded_ana - virtual_ana) / commanded_ana) * 100
        else:
            power_loss_rate = 0.0
            
        # 15% 이상 힘이 증발했다면 경고 출력
        if power_loss_rate > 5.0:
            robot_num = meta_info['Robot_Num']
            tick = meta_info['Tick_Index']
            msg = f"🚨 [로봇 {robot_num}번 / {tick}틱] 전력 손실 감지! (명령: {commanded_ana:.0f} vs 가상 역산: {virtual_ana:.0f} | 손실률: {power_loss_rate:.1f}%)"
            print(msg)
            loss_alerts.append(msg)

print("\n" + "="*60)
print(f"✅ 가상 센서 테스트 완료! 총 {len(records)}틱 중 {len(loss_alerts)}회의 전력 손실 알람 발생.")
print("="*60)

🚀 가상 센서 (Virtual Sensor) 단독 테스트 시작...

✅ 가상 센서 모델 로드 완료!
[Unknown Time] [틱 00] 명령 Ana: 10935 | 가상 역산: 10934
[Unknown Time] [틱 01] 명령 Ana: 10935 | 가상 역산: 10935
[Unknown Time] [틱 02] 명령 Ana: 10935 | 가상 역산: 10935
[Unknown Time] [틱 03] 명령 Ana: 10935 | 가상 역산: 10936
[Unknown Time] [틱 00] 명령 Ana: 9017 | 가상 역산: 9001
[Unknown Time] [틱 00] 명령 Ana: 9017 | 가상 역산: 9039
[Unknown Time] [틱 01] 명령 Ana: 9017 | 가상 역산: 8993
[Unknown Time] [틱 00] 명령 Ana: 9464 | 가상 역산: 9371
[Unknown Time] [틱 01] 명령 Ana: 9464 | 가상 역산: 9467
[Unknown Time] [틱 02] 명령 Ana: 9464 | 가상 역산: 9466
[Unknown Time] [틱 03] 명령 Ana: 9464 | 가상 역산: 9464
[Unknown Time] [틱 00] 명령 Ana: 10975 | 가상 역산: 10738
[Unknown Time] [틱 01] 명령 Ana: 10975 | 가상 역산: 10941
[Unknown Time] [틱 02] 명령 Ana: 10975 | 가상 역산: 10975
[Unknown Time] [틱 03] 명령 Ana: 10975 | 가상 역산: 10954
[Unknown Time] [틱 00] 명령 Ana: 9017 | 가상 역산: 8970
[Unknown Time] [틱 01] 명령 Ana: 9017 | 가상 역산: 9017
[Unknown Time] [틱 02] 명령 Ana: 9017 | 가상 역산: 9014
[Unknown Time] [틱 03] 명령 Ana: 9017 | 가상 역산: 9

In [11]:
fast_ex.save_to_csv(raw_valid, save_dir='./Ana_csv')

💾 [저장 완료] 분석용 CSV 파일이 생성되었습니다: ./Ana_csv/2026-04-15_110735_analysis.csv


In [13]:
raw_valid

,Pump_BuildUp_P1,RB1_Robot_Num,g_s_SV_P1,Ana_Out_P1,Scale_Out___PT_P1,Scale_Out___FT_P1,TK_Temp_PV_P1,TK_Level_PV_P1
Time,,,,,,,,
2026-04-14 10:30:42.651238,1.0,37.0,2680.0,10935.0,122.0,1795.0,NaN,NaN
2026-04-14 10:30:43.367209,1.0,37.0,2680.0,10935.0,127.0,2687.0,NaN,NaN
2026-04-14 10:30:45.145280,1.0,37.0,2680.0,10935.0,124.0,2687.0,NaN,NaN
2026-04-14 10:30:47.993071,1.0,37.0,2680.0,10935.0,126.0,2687.0,NaN,NaN
2026-04-14 10:30:50.928229,0.0,37.0,2680.0,5333.0,66.0,1534.0,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-04-15 10:30:29.358077,1.0,29.0,2340.0,9464.0,105.0,2340.0,21.0,58.0
2026-04-15 10:30:31.369717,1.0,29.0,2340.0,9464.0,108.0,2337.0,21.0,58.0
2026-04-15 10:30:33.368898,0.0,29.0,2340.0,5333.0,68.0,1548.0,21.0,58.0
